In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Trips

In [3]:
def total_trips(data1, data2, tag='PSRC Region'):
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    Trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])

    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = Trip_1_total
    tpp[f'{survey_year}Survey'] = Trip_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.0f}',
        f'{survey_year}Survey': '{:,.0f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.0f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [4]:
total_trips(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,"16,277,616","13,902,503","2,375,113",17.1%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_trips(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,"1,272,126","853,837","418,289",49.0%


## Trips per Person

In [6]:
def trip_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    Trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])

    ##Trips per person
    tpp1 = Trip_1_total / Person_1_total
    tpp2 = Trip_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [7]:
trip_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,3.8,3.7,0.1,2.8%


In [8]:
trip_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,3.9,3.0,1.0,32.8%


## Trips per Person by Purpose

In [9]:
def trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'dpurp', 'Trip Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.1f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Purpose',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [10]:
trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Purpose,,,,
Work,0.5,0.5,-0.0,-0.5%
School,0.1,0.1,0.0,2.8%
Escort,0.4,0.4,0.0,0.5%
Personal Business,0.2,0.2,-0.0,-14.0%
Shop,0.5,0.5,-0.0,-0.5%
Meal,0.2,0.2,0.0,3.9%
Social,0.5,0.5,0.1,9.4%


In [11]:
trips_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Purpose,,,,
Work,0.5,0.4,0.1,24.4%
School,0.2,0.1,0.0,16.5%
Escort,0.4,0.3,0.1,16.4%
Personal Business,0.2,0.1,0.0,21.8%
Shop,0.5,0.2,0.2,110.1%
Meal,0.3,0.3,0.0,9.9%
Social,0.6,0.5,0.2,37.0%


## Trips per Person by Mode

In [12]:
def trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'mode', 'Trip Mode')
    tpbp = tpbp.loc[trip_mode_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.1f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Mode',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [13]:
trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Mode,,,,
Walk,0.6,0.4,0.1,37.1%
Bike,0.0,0.0,-0.0,-80.3%
SOV,1.8,1.7,0.1,8.9%
HOV2,0.9,0.8,0.1,13.8%
HOV3+,0.4,0.6,-0.2,-28.7%
Transit,0.1,0.1,-0.0,-40.6%
School Bus,0.0,0.1,-0.0,-30.3%


In [14]:
trips_per_ps_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Mode,,,,
Walk,0.5,0.4,0.1,18.4%
Bike,0.0,0.0,-0.0,-73.3%
SOV,1.9,1.2,0.7,58.7%
HOV2,1.0,0.7,0.3,38.6%
HOV3+,0.4,0.4,0.0,9.2%
Transit,0.1,0.1,-0.0,-17.5%
School Bus,0.1,0.1,-0.0,-26.6%


## Trip Share by Purpose

In [15]:
def pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Purpose
    trip_1_total = get_total(data_daysim['Trip']['trexpfac'])
    trip_2_total = get_total(data_survey['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'dpurp', 'Trips Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Purpose',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [16]:
pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Purpose,,,,
Work,12.8%,13.2%,-0.4%,-3.2%
School,3.9%,3.9%,-0.0%,-0.1%
Escort,9.7%,9.9%,-0.2%,-2.3%
Personal Business,4.6%,5.5%,-0.9%,-16.3%
Shop,13.0%,13.4%,-0.4%,-3.3%
Meal,6.7%,6.6%,0.1%,1.1%
Social,14.3%,13.4%,0.9%,6.4%


In [17]:
pc_trip_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Purpose,,,,
Work,0.9%,0.8%,0.1%,19.2%
School,0.3%,0.3%,0.0%,11.7%
Escort,0.7%,0.6%,0.1%,11.5%
Personal Business,0.3%,0.3%,0.1%,16.7%
Shop,0.9%,0.5%,0.5%,101.3%
Meal,0.6%,0.6%,0.0%,5.3%
Social,1.2%,0.9%,0.3%,31.2%


## Trip Share by Mode

In [18]:
def pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Mode
    trip_1_total = get_total(data_daysim['Trip']['trexpfac'])
    trip_2_total = get_total(data_survey['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'mode', 'Trips Mode')
    ptbp = ptbp.loc[trip_mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Mode',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [19]:
pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Mode,,,,
Walk,14.9%,11.2%,3.7%,33.3%
Bike,0.2%,1.2%,-1.0%,-80.8%
SOV,48.3%,45.6%,2.7%,5.9%
HOV2,23.1%,20.9%,2.2%,10.7%
HOV3+,10.6%,15.3%,-4.7%,-30.7%
Transit,1.7%,3.0%,-1.2%,-42.3%
School Bus,1.1%,1.6%,-0.5%,-32.2%


In [20]:
pc_trip_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Mode,,,,
Walk,1.0%,0.9%,0.1%,13.4%
Bike,0.0%,0.1%,-0.0%,-74.5%
SOV,3.8%,2.5%,1.3%,52.1%
HOV2,1.9%,1.4%,0.5%,32.8%
HOV3+,0.8%,0.8%,0.0%,4.6%
Transit,0.2%,0.3%,-0.1%,-20.9%
School Bus,0.1%,0.1%,-0.0%,-29.6%


## Trip Distance by Purpose

In [21]:
def trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [22]:
trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,9.0,9.2,-0.2,-2.4%
School,4.7,3.9,0.8,19.2%
Escort,5.9,5.7,0.2,2.9%
Personal Business,5.3,5.6,-0.3,-6.2%
Shop,4.3,4.4,-0.0,-0.9%
Meal,3.8,3.3,0.5,14.7%
Social,5.2,6.7,-1.5,-22.0%


In [23]:
trips_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,7.6,6.7,0.9,13.2%
School,3.8,4.2,-0.4,-9.6%
Escort,4.8,7.5,-2.7,-36.3%
Personal Business,5.0,4.2,0.8,18.5%
Shop,3.0,4.2,-1.1,-27.3%
Meal,3.1,2.9,0.2,7.4%
Social,4.4,6.4,-2.0,-31.8%


## Trip Distance by Mode

In [24]:
def trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl = atl.loc[trip_mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [25]:
trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Mode,,,,
Walk,0.9,0.9,-0.0,-0.3%
Bike,4.6,2.0,2.6,130.4%
SOV,7.0,6.9,0.1,1.5%
HOV2,5.9,6.2,-0.3,-5.2%
HOV3+,6.8,7.3,-0.6,-7.7%
Transit,9.7,8.9,0.8,9.4%
School Bus,3.8,3.6,0.2,5.2%


In [26]:
trips_distance_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Mode,,,,
Walk,1.4,0.7,0.7,100.6%
Bike,7.5,1.5,6.0,407.0%
SOV,5.4,6.2,-0.8,-12.9%
HOV2,4.5,8.5,-4.0,-47.5%
HOV3+,4.5,5.4,-0.9,-16.5%
Transit,5.5,10.2,-4.7,-46.1%
School Bus,2.8,3.1,-0.4,-11.6%


## Trip Travel Time by Purpose

In [27]:
def trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [28]:
trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Purpose,,,,
Work,24.5,21.6,2.9,13.3%
School,18.5,12.4,6.1,49.0%
Escort,19.2,14.7,4.5,30.9%
Personal Business,18.6,21.6,-3.1,-14.1%
Shop,17.0,12.7,4.3,34.0%
Meal,15.2,12.4,2.8,22.7%
Social,18.8,17.2,1.5,8.8%


In [29]:
trips_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Purpose,,,,
Work,18.1,17.8,0.3,1.6%
School,14.7,11.8,2.8,23.9%
Escort,13.1,17.2,-4.1,-23.9%
Personal Business,14.3,13.2,1.1,8.4%
Shop,11.1,12.0,-0.9,-7.3%
Meal,11.1,9.3,1.8,18.9%
Social,14.0,19.2,-5.2,-26.9%


## Trip Travel Time by Mode

In [30]:
def trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[trip_mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [31]:
trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Mode,,,,
Walk,18.1,18.2,-0.1,-0.8%
Bike,30.9,12.1,18.8,155.5%
SOV,20.3,16.1,4.2,25.7%
HOV2,18.1,14.5,3.6,24.6%
HOV3+,19.8,16.3,3.5,21.8%
Transit,29.5,56.0,-26.5,-47.3%
School Bus,12.7,10.1,2.5,25.1%


In [32]:
trips_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Mode,,,,
Walk,28.6,14.4,14.2,99.1%
Bike,49.9,8.9,41.0,462.0%
SOV,11.8,15.1,-3.2,-21.5%
HOV2,9.7,17.6,-7.9,-45.0%
HOV3+,9.5,13.4,-3.8,-28.7%
Transit,25.6,27.8,-2.2,-7.9%
School Bus,6.3,8.5,-2.2,-25.6%


## Trips by District

In [33]:
##Trips per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Trip'] = data1['Trip'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Trip_cloned'] = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Trip'].groupby(by='DistrictFlowName')['trexpfac'].sum()
todist2 = data2['Trip_cloned'].groupby(by='DistrictFlowName')['trexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

,DaysimOutputs,2023Survey,Difference,% Difference
DistrictFlowName,,,,
Bellevue (excluding downtown),"538,423","426,535","111,888",26.2%
Bellevue Downtown,"72,569","19,581","52,988",270.6%
Kirkland,"370,335","156,492","213,843",136.6%
Redmond,"290,799","251,229","39,570",15.8%
Seattle (excluding Seattle downtown),"2,827,196","2,052,351","774,845",37.8%
Seattle downtown,"397,768","221,974","175,794",79.2%
Rest,"11,780,526","8,337,507","3,443,019",41.3%


In [34]:
def trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    display(trip_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (trip_by_district_purpose1 - trip_by_district_purpose2) / trip_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [35]:
def trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    trip_by_district_purpose1 = trip_by_district_purpose1.div(trip_by_district_purpose1.sum(axis=0), axis=1) * 100
    trip_by_district_purpose2 = trip_by_district_purpose2.div(trip_by_district_purpose2.sum(axis=0), axis=1) * 100
    display(trip_by_district_purpose1.style.format('{:,.1f}%').set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format('{:,.1f}%').set_caption(f"{survey_year}Survey"))

## Trips by District by Purpose

In [36]:
trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='dpurp', gp2='DistrictFlowName', 
                        gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,"58,691","7,897","45,237","35,619","419,903","73,052","1,444,501"
School,"26,954","1,969","15,007","11,804","109,726","6,298","455,870"
Escort,"51,223","4,282","33,816","26,898","234,676","19,050","1,202,155"
Personal Business,"22,345","3,854","15,925","12,242","122,849","18,303","552,967"
Shop,"61,718","9,477","42,808","33,362","314,636","44,005","1,605,272"
Meal,"41,950","6,763","29,213","22,849","222,154","36,130","727,337"
Social,"83,980","11,690","57,510","45,129","414,051","57,376","1,658,623"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,"48,404","2,105","22,413","32,729","284,045","24,064","1,071,449"
School,"17,564",485,"6,663","17,917","87,425","1,864","373,969"
Escort,"52,885",309,"16,923","18,872","214,830","17,988","840,813"
Personal Business,"23,743","1,357","6,131","8,556","86,163","9,377","503,748"
Shop,"29,946","2,997","20,814","8,757","226,225","32,539","1,231,062"
Meal,"54,027","2,906","12,741","12,070","128,974","23,421","479,709"
Social,"61,144","2,459","13,513","51,942","286,611","46,249","1,020,379"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,21.3%,275.2%,101.8%,8.8%,47.8%,203.6%,34.8%
School,53.5%,306.1%,125.2%,-34.1%,25.5%,237.9%,21.9%
Escort,-3.1%,"1,283.9%",99.8%,42.5%,9.2%,5.9%,43.0%
Personal Business,-5.9%,184.1%,159.8%,43.1%,42.6%,95.2%,9.8%
Shop,106.1%,216.2%,105.7%,281.0%,39.1%,35.2%,30.4%
Meal,-22.4%,132.7%,129.3%,89.3%,72.2%,54.3%,51.6%
Social,37.3%,375.4%,325.6%,-13.1%,44.5%,24.1%,62.5%


In [37]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='dpurp', gp2='DistrictFlowName', 
                        gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,16.9%,17.2%,18.9%,19.0%,22.8%,28.7%,18.9%
School,7.8%,4.3%,6.3%,6.3%,6.0%,2.5%,6.0%
Escort,14.8%,9.3%,14.1%,14.3%,12.8%,7.5%,15.7%
Personal Business,6.4%,8.4%,6.6%,6.5%,6.7%,7.2%,7.2%
Shop,17.8%,20.6%,17.9%,17.8%,17.1%,17.3%,21.0%
Meal,12.1%,14.7%,12.2%,12.2%,12.1%,14.2%,9.5%
Social,24.2%,25.5%,24.0%,24.0%,22.5%,22.6%,21.7%


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,16.8%,16.7%,22.6%,21.7%,21.6%,15.5%,19.4%
School,6.1%,3.8%,6.7%,11.9%,6.7%,1.2%,6.8%
Escort,18.4%,2.5%,17.1%,12.5%,16.3%,11.6%,15.2%
Personal Business,8.3%,10.8%,6.2%,5.7%,6.6%,6.0%,9.1%
Shop,10.4%,23.8%,21.0%,5.8%,17.2%,20.9%,22.3%
Meal,18.8%,23.0%,12.8%,8.0%,9.8%,15.1%,8.7%
Social,21.3%,19.5%,13.6%,34.4%,21.8%,29.7%,18.5%


## Trips by District by Mode

In [38]:
trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='mode', 
                    gp1_label='DistrictFlowName', gp2_label='Trip Mode',
                    gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),"56,834",832,"256,178","138,409","60,449","17,297","8,424"
Bellevue Downtown,"32,154",373,"22,007","10,861","3,107","3,634",433
Kirkland,"38,459",606,"187,588","92,885","37,002","9,759","4,036"
Redmond,"33,944",669,"144,884","70,578","29,520","8,009","3,195"
Seattle (excluding Seattle downtown),"701,144","9,075","1,261,811","515,120","212,855","102,703","24,488"
Seattle downtown,"251,169","2,060","88,833","29,433","8,895","16,975",403
Rest,"1,312,054","24,341","5,901,475","2,911,116","1,380,139","119,386","132,015"


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),"76,702","4,485","147,254","128,070","54,563","7,128","2,265"
Bellevue Downtown,"4,578",442,"7,211","5,947",523,694,nan
Kirkland,"23,580",nan,"83,706","21,964","17,462","5,945","1,992"
Redmond,"16,671","3,366","104,779","45,137","33,636","28,039","15,273"
Seattle (excluding Seattle downtown),"393,006","76,872","836,632","433,242","191,892","96,273","13,117"
Seattle downtown,"89,162","7,743","49,583","45,357","9,335","14,781",55
Rest,"556,911","38,913","3,946,708","1,796,021","1,514,071","206,229","154,794"


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),-25.9%,-81.5%,74.0%,8.1%,10.8%,142.7%,272.0%
Bellevue Downtown,602.4%,-15.5%,205.2%,82.6%,494.4%,423.3%,nan%
Kirkland,63.1%,nan%,124.1%,322.9%,111.9%,64.2%,102.6%
Redmond,103.6%,-80.1%,38.3%,56.4%,-12.2%,-71.4%,-79.1%
Seattle (excluding Seattle downtown),78.4%,-88.2%,50.8%,18.9%,10.9%,6.7%,86.7%
Seattle downtown,181.7%,-73.4%,79.2%,-35.1%,-4.7%,14.8%,633.0%
Rest,135.6%,-37.4%,49.5%,62.1%,-8.8%,-42.1%,-14.7%


In [39]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='mode', 
                    gp1_label='DistrictFlowName', gp2_label='Trip Mode',
                    gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),2.3%,2.2%,3.3%,3.7%,3.5%,6.2%,4.9%
Bellevue Downtown,1.3%,1.0%,0.3%,0.3%,0.2%,1.3%,0.3%
Kirkland,1.6%,1.6%,2.4%,2.5%,2.1%,3.5%,2.3%
Redmond,1.4%,1.8%,1.8%,1.9%,1.7%,2.9%,1.8%
Seattle (excluding Seattle downtown),28.9%,23.9%,16.0%,13.7%,12.3%,37.0%,14.2%
Seattle downtown,10.4%,5.4%,1.1%,0.8%,0.5%,6.1%,0.2%
Rest,54.1%,64.1%,75.1%,77.3%,79.7%,43.0%,76.3%


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),6.6%,3.4%,2.8%,5.2%,3.0%,2.0%,1.2%
Bellevue Downtown,0.4%,0.3%,0.1%,0.2%,0.0%,0.2%,nan%
Kirkland,2.0%,nan%,1.6%,0.9%,1.0%,1.7%,1.1%
Redmond,1.4%,2.6%,2.0%,1.8%,1.8%,7.8%,8.1%
Seattle (excluding Seattle downtown),33.9%,58.3%,16.2%,17.5%,10.5%,26.8%,7.0%
Seattle downtown,7.7%,5.9%,1.0%,1.8%,0.5%,4.1%,0.0%
Rest,48.0%,29.5%,76.3%,72.5%,83.1%,57.4%,82.6%
